# 01 - EDA And Validation Design

This notebook goes one level deeper.

Goals:
- inspect the dataset by year and sector
- think about what a realistic validation split should be
- build trivial baselines that future models must beat

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_squared_error
from IPython.display import display

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 100)

ROOT = Path.cwd().resolve().parent
DATA_DIR = ROOT / 'data' / 'raw'

train = pd.read_csv(
    DATA_DIR / 'train.csv',
    parse_dates=['period_start', 'period_end'],
)

train['obs_year'] = train['period_start'].dt.year
train['obs_quarter'] = train['period_start'].dt.quarter

train.shape

In [ ]:
year_counts = train.groupby('obs_year')['id'].count().rename('rows')
ticker_counts = train.groupby('obs_year')['ticker'].nunique().rename('unique_tickers')
summary_by_year = pd.concat([year_counts, ticker_counts], axis=1)
display(summary_by_year)

if 'sector_code' in train.columns:
    sector_year = pd.crosstab(train['obs_year'], train['sector_code'])
    display(sector_year)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.violinplot(data=train, x='obs_year', y='return_pct', inner='quartile', ax=axes[0])
axes[0].set_title('Target By Observation Year')
axes[0].set_xlabel('Observation year')
axes[0].set_ylabel('return_pct')

clip_low = train['return_pct'].quantile(0.01)
clip_high = train['return_pct'].quantile(0.99)
clipped = train['return_pct'].clip(clip_low, clip_high)
sns.histplot(clipped, bins=60, ax=axes[1])
axes[1].set_title('Target Distribution (1st to 99th pct clipped view)')
axes[1].set_xlabel('return_pct')

plt.tight_layout()
plt.show()

In [ ]:
missing_by_year = (
    train.groupby('obs_year')
    .apply(lambda frame: frame.isna().mean())
    .T
)

top_variable_features = (
    missing_by_year.max(axis=1) - missing_by_year.min(axis=1)
).sort_values(ascending=False)

display(top_variable_features.head(15).rename('missingness_range'))

selected_features = top_variable_features.head(10).index.tolist()
display(missing_by_year.loc[selected_features])

## Baseline Validation Choice

A simple, defensible split is:

- train on observations before `2022-01-01`
- validate on observations from `2022-01-01` onward

That is not perfect, but it is much better than a random split and is easy for the whole team to reproduce.

In [ ]:
train_mask = train['period_start'] < '2022-01-01'
valid_mask = ~train_mask

train_fold = train.loc[train_mask].copy()
valid_fold = train.loc[valid_mask].copy()

print('train fold:', train_fold.shape)
print('valid fold:', valid_fold.shape)
print('train period range:', train_fold['period_start'].min(), 'to', train_fold['period_start'].max())
print('valid period range:', valid_fold['period_start'].min(), 'to', valid_fold['period_start'].max())

In [ ]:
def rmse(y_true, y_pred):
    return mean_squared_error(y_true, y_pred, squared=False)

global_median = train_fold['return_pct'].median()
median_pred = np.full(len(valid_fold), global_median)

print('Global median baseline RMSE:', round(rmse(valid_fold['return_pct'], median_pred), 4))

if 'sector_code' in train.columns:
    sector_medians = train_fold.groupby('sector_code')['return_pct'].median()
    sector_pred = valid_fold['sector_code'].map(sector_medians).fillna(global_median)
    print('Sector median baseline RMSE:', round(rmse(valid_fold['return_pct'], sector_pred), 4))

In [ ]:
folds = []
for valid_year in [2020, 2021, 2022]:
    fold_train = train[train['obs_year'] < valid_year]
    fold_valid = train[train['obs_year'] == valid_year]
    folds.append(
        {
            'valid_year': valid_year,
            'train_rows': len(fold_train),
            'valid_rows': len(fold_valid),
            'unique_train_tickers': fold_train['ticker'].nunique(),
            'unique_valid_tickers': fold_valid['ticker'].nunique(),
        }
    )

display(pd.DataFrame(folds))

## Recommended Takeaways

- Use a time-aware split as the default standard.
- Benchmark every model against trivial baselines.
- Track whether performance is stable across years, not just on one holdout.
- Treat sector effects and missingness as potential signal sources.

Next: open `02_baseline_models_and_submission.ipynb` to build a first real submission.